# Linkedin


In [ ]:
pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [ ]:
!pip install numpy==1.26.4

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import tool

In [ ]:
import openai

openai.api_key = " "

In [ ]:
import os
#from utils import get_openai_api_key

#openai_api_key = get_openai_api_key()
import os
os.environ["OPENAI_API_KEY"] = " "
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

In [ ]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

In [ ]:
import os
os.environ["SERPER_API_KEY"] = " "

## Creating Agents for LinkedIn

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

In [ ]:
planner = Agent(
    role="LinkedIn Content Planner",
    goal="Plan engaging and factually accurate content for LinkedIn on {topic}",
    backstory="You're working on planning a LinkedIn post "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
	  verbose=True
)

### Agent: Writer

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic for LinkedIn: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic for LinkedIn: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements"
              "Make it engaging, use bullet points where needed, and end with a question to encourage comments and also make sure content is not more than 500 words"
              "You are a LinkedIn content expert who can write posts that go viral",
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a LinkedIn article "
              "from the Content Writer. "
              "Your goal is to review the LinkedIn article "
              "to ensure that it follows  best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible"
              "Fix grammar, improve the flow, make the tone more natural, and ensure it sounds human and engaging"
              "You are a meticulous editor, polishing posts to sound engaging and professional.",

    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"

    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis "
        " and resources.",
    agent=planner,
)

### Task: Write

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "LinkedIn post on {topic}.\n"
        "2. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion and content is not more than 500 words\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice and add relevant hashtags.\n"
    ),
    expected_output="A well-structured LinkedIn post with hook, insights, and call-to-action.",
    agent=writer,
)

### Task: Edit

In [ ]:
edit = Task(
    description=("Proofread the given LinkedIn post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A final LinkedIn post ready to publish.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Multi-Agent Framework"})

- Display the results of your execution as markdown in the notebook.

In [ ]:
from IPython.display import Markdown
Markdown(result)